# 🧱 NYC Yellow Taxi Analytics — Landing to Bronze Synchronizer
---
### 👤 Developer Profile
* **Name:** `cacciari@gmail.com`
* **Contact:** [cacciari@gmail.com](mailto:cacciari@gmail.com)
* **LinkedIn:** [linkedin.com/in/cacciari](https://linkedin.com/in/cacciari)

### 📋 Module Overview
* **File Name:** `02_landing_zone_to_bronze.py`
* **Target Layer:** `Bronze Layer (Raw Storage)`
* **Short Description:** Append-only ingestion job mapping incoming schema files to Delta Lake format. Implements `overwriteSchema` and `mergeSchema` structures to ensure seamless schema evolution over chronological inputs while embedding systemic metadata and tracking columns.

In [0]:
%skip
%pip install datacompy
%pip install datacompy[spark]


In [0]:
dbutils.library.restartPython()

#### Imports

In [0]:

import sys
import os 
sys.path.append(os.path.abspath('..'))
from modules.data_ingestion import ingest_data, create_metadata_dict, generate_schema_matrix
from pyspark.sql.functions import current_timestamp, input_file_name, col, lit, first
from pyspark.sql.types import StructType, StructField, StringType
from typing import Dict, List, Any
from constants import (
    METADATA_TYPE_COLUMNS, LANDING_ZONE_FILES, LANDING_ZONE_DIRECTORY, LANDING_ZONE_PATH, LANDING_ZONE_FORMAT, BRONZE_TABLE, TARGET_RAW_COLUMNS, AUDIT_COLUMNS
    )


In [0]:
print(METADATA_TYPE_COLUMNS)

#### Functions

#### Pipeline configuration

In [0]:
AUDIT_COLUMNS["AUD_ING_LDZ_TO_BRZ_SCRIPT"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get().split('/')[-1]

# WIDGET
dbutils.widgets.text("BRONZE_TABLE", BRONZE_TABLE)


#### Pre-Bronze Ingestion Checks
* Show ingestion parameters
* Create a dataframe set
* Create a dataframe metadata set for quality control
* Correct types accordingly (method: manual decision)

In [0]:
print("Parameter Set:")
print(f"\tLanding Zone Source Files: {LANDING_ZONE_FILES}")
print(f"\tLanding Zone Directory: {LANDING_ZONE_DIRECTORY}")
print(f"\tLanding Zone Path: {LANDING_ZONE_PATH}")
# Print file names in Landing Zone Directory

files_list = []
print(f"Files in Landing Zone Directory:")
for file in dbutils.fs.ls(LANDING_ZONE_DIRECTORY):
    if file.name.endswith(LANDING_ZONE_FORMAT):
        files_list.append(file.name)
        print(f"\t\t{file.name}")

# Create Dataframe dictionaries
df_dict={} # Dictionary of Dataframes
df_meta_dict={} # Dictionary of Dataframe metadata

print("Creating Raw Dataframe:")
for file in dbutils.fs.ls(LANDING_ZONE_DIRECTORY):
    if file.name.endswith(LANDING_ZONE_FORMAT):
        try:
            #Acquire only columns of interest as in TARGET_RAW_COLUMNS 
            df_raw = spark.read.parquet(file.path)
            # df_dict[file.name] = df_raw.select(*TARGET_RAW_COLUMNS)
            df_dict[file.name] = df_raw
            print(f"\t{file.name} is a valid parquet file")
        except:
            print(f"\t{file.name} is not a valid parquet file")

##### Schema Harmonization
* Correct types accordingly (method: manual decision)

In [0]:
help(ingest_data)

In [0]:
# Metadata
df_meta_dict = create_metadata_dict(df_dict, METADATA_TYPE_COLUMNS)
# Creating type Matrix
df_schema_matrix = generate_schema_matrix(df_meta_dict, files_list=files_list)
display(df_schema_matrix)

In [0]:
# After analysis, let´s correct January

# There's a special case in column for airport fees
for file_name, df in df_dict.items():
    if "Airport_fee" in df.columns:
        df_dict[file_name] = df_dict[file_name].withColumnRenamed("Airport_fee", "airport_fee")
        print(f"\t{file_name} renamed column Airport_fee to airport_fee")
    # if "airport_fee" not in df.columns:
    #     df_dict[file_name] = df_dict[file_name].withColumn("airport_fee", lit(None).cast("double"))
    #     print(f"\t{file_name} added column airport_fee")

# Cast January to correct types
df_dict[files_list[0]] = df_dict[files_list[0]] \
    .withColumn("VendorID", col("VendorID").cast("int")) \
    .withColumn("passenger_count", col("passenger_count").cast("long")) \
    .withColumn("DOLocationID", col("DOLocationID").cast("int")) \
    .withColumn("PULocationID", col("PULocationID").cast("int")) \
    .withColumn("RatecodeID", col("RatecodeID").cast("long")) \
    
print("Schema harmonization done. Rechecking")
# Metadata
df_meta_dict = create_metadata_dict(df_dict, METADATA_TYPE_COLUMNS)
# Creating type Matrix
df_schema_matrix = generate_schema_matrix(df_meta_dict, files_list=files_list)
display(df_schema_matrix)

#### Ingest to Bronze
* Ingest data to bronze table

In [0]:
# Stack Dataframes for bronze layer
AUDIT_COLUMNS["CURRENT_SCRIPT"] = AUDIT_COLUMNS["AUD_ING_LDZ_TO_BRZ_SCRIPT"]
if ingest_data(df_dict, BRONZE_TABLE, AUDIT_COLUMNS ):
    print("Data successfully ingested")
else:
    print("Data ingestion failed")

In [0]:
%sql
    -- SELECT * FROM IDENTIFIER(:BRONZE_TABLE)
    SELECT * FROM ifood.bronze.BRZ_TXI_YEL_TRIP_RAW LIMIT 10;